In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/blessingtosin/cc-approvals-dataset/cc_approvals.data


In [2]:
# Loading dataset
cc_app = pd.read_csv("/kaggle/input/datasets/blessingtosin/cc-approvals-dataset/cc_approvals.data")
cc_app.head()

,b,30.83,0.0,u,g,w,v,1.25,t,t.1,1,g.1,0,+
0,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
1,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
2,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
3,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+
4,b,32.08,4.000,u,g,m,v,2.50,t,f,0,g,0,+


In [3]:
# Checking data information
cc_app.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 689 entries, 0 to 688
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   b       689 non-null    object 
 1   30.83   689 non-null    object 
 2   0.0     689 non-null    float64
 3   u       689 non-null    object 
 4   g       689 non-null    object 
 5   w       689 non-null    object 
 6   v       689 non-null    object 
 7   1.25    689 non-null    float64
 8   t       689 non-null    object 
 9   t.1     689 non-null    object 
 10  1       689 non-null    int64  
 11  g.1     689 non-null    object 
 12  0       689 non-null    int64  
 13  +       689 non-null    object 
dtypes: float64(2), int64(2), object(10)
memory usage: 75.5+ KB


In [4]:
# Checking for missing values
cc_app.isnull().sum()

b        0
30.83    0
0.0      0
u        0
g        0
w        0
v        0
1.25     0
t        0
t.1      0
1        0
g.1      0
0        0
+        0
dtype: int64

In [5]:
# Making s copy fron the original dataset
df = cc_app.copy()

In [6]:
# Splitting into feature and target
y = df['+']
df = df.drop(columns=['+'])

In [7]:
# Seperating the dataset into categorical columns and numetical columns
cat_cols = []
num_cols = []

for col in df.columns:
    if df[col].dtype == 'object':
        cat_cols.append(col)
    else:
        num_cols.append(col)

In [8]:
# Encoding categorical xolumns

for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

In [9]:
# scaling numetical columns
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [10]:
X = df

In [11]:
X.head()

,b,30.83,0.0,u,g,w,v,1.25,t,t.1,1,g.1,0
0,1,327,-0.061435,2,1,11,4,0.243606,1,1,0.739920,0,-0.088074
1,1,89,-0.857438,2,1,11,4,-0.216602,1,0,-0.493976,0,-0.037402
2,2,125,-0.648387,2,1,13,8,0.455780,1,1,0.534270,0,-0.194985
3,2,43,0.172742,2,1,13,8,-0.153847,1,0,-0.493976,2,-0.195561
4,2,167,-0.153900,2,1,10,8,0.082234,1,0,-0.493976,0,-0.195561


In [12]:
y.head()

0    +
1    +
2    +
3    +
4    +
Name: +, dtype: object

**DATA SPLITING (TRAIN AND TEST)**

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=9)

In [14]:
X_train.head()

,b,30.83,0.0,u,g,w,v,1.25,t,t.1,1,g.1,0
107,2,241,-0.505669,3,3,14,4,2.323511,1,0,-0.493976,0,-0.195561
537,2,188,-0.606175,2,1,13,1,-0.590149,1,0,-0.493976,0,-0.195561
518,2,228,-0.614215,2,1,14,8,-0.627503,1,1,0.534270,0,-0.195561
50,2,106,-0.756933,2,1,11,8,-0.141893,1,0,-0.493976,0,-0.195561
478,0,111,-0.413204,3,3,0,0,-0.639457,0,0,-0.493976,2,-0.195561


In [15]:
y_train.head()

107    -
537    -
518    +
50     +
478    -
Name: +, dtype: object

**BUILDING PIPELINE FOR EACH MODEL**

In [16]:

logistic_pipeline =  Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
]) 

decision_tree_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeClassifier())
]) 

random_forest_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

**HYPERPARAMETER TUNING**

In [17]:
# Hyperparameter tuning to find the best setting for each model

logistic_params = {
    'model__penalty': ['l1', 'l2', 'elasticnet'],
    'model__C': [0.01, 0.1, 1, 10],
    'model__solver': ['liblinear', 'saga']
}

decision_tree_params = {
    'model__max_depth': [3, 5, 10],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

random_forest_params = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [5, 10]
}

**HYPERPARAMETER TUNING WITH GRIDSEARCHCV**

In [18]:
logistic_grid = GridSearchCV(logistic_pipeline, logistic_params)

decision_grid = GridSearchCV(decision_tree_pipeline, decision_tree_params)

random_grid = GridSearchCV(random_forest_pipeline, random_forest_params)

In [19]:
# Training and testing the model (logistic regression) using GridSearchCV

logistic_grid.fit(X_train, y_train)
y_pred_log = logistic_grid.predict(X_test)
print(y_pred_log)

['-' '-' '+' '-' '+' '+' '+' '-' '-' '-' '-' '+' '+' '-' '-' '-' '-' '-'
 '+' '+' '+' '+' '+' '-' '+' '-' '-' '+' '+' '-' '-' '+' '-' '-' '-' '+'
 '-' '-' '+' '+' '+' '+' '-' '+' '-' '-' '+' '-' '+' '+' '-' '+' '-' '+'
 '-' '-' '-' '+' '-' '-' '+' '-' '+' '-' '-' '-' '-' '-' '-' '+' '-' '+'
 '-' '+' '-' '+' '-' '-' '-' '+' '-' '+' '+' '+' '+' '-' '-' '+' '+' '+'
 '-' '-' '+' '-' '+' '+' '+' '-' '+' '+' '+' '-' '+' '+' '-' '+' '-' '-'
 '-' '+' '+' '-' '-' '+' '+' '-' '+' '+' '+' '-' '+' '+' '+' '-' '+' '-'
 '-' '+' '-' '-' '+' '-' '-' '+' '+' '-' '-' '+']


In [20]:
# Training and testing the model (decision tree) using GridSearchCV

decision_grid.fit(X_train, y_train)
y_pred_decision = decision_grid.predict(X_test)
print(y_pred_decision)

['-' '-' '+' '-' '+' '+' '+' '-' '-' '-' '-' '+' '-' '-' '-' '-' '-' '-'
 '+' '+' '+' '+' '+' '-' '+' '-' '-' '+' '+' '-' '-' '+' '-' '-' '-' '+'
 '-' '-' '+' '+' '+' '+' '+' '+' '-' '-' '-' '-' '-' '+' '-' '+' '-' '-'
 '-' '-' '+' '+' '-' '-' '+' '+' '+' '-' '-' '-' '-' '-' '-' '+' '-' '+'
 '-' '+' '-' '+' '-' '-' '-' '+' '+' '-' '+' '+' '+' '-' '+' '+' '+' '+'
 '-' '-' '+' '-' '+' '+' '+' '-' '+' '+' '+' '-' '+' '+' '-' '+' '+' '-'
 '-' '-' '+' '-' '-' '+' '-' '-' '+' '+' '+' '-' '+' '-' '+' '-' '+' '-'
 '-' '+' '-' '-' '-' '+' '-' '-' '+' '-' '-' '+']


In [21]:
# Training and testing the model (random forest) using GridSearchCV

random_grid.fit(X_train, y_train)
y_pred_random = random_grid.predict(X_test)
print(y_pred_random)

['-' '-' '+' '-' '+' '+' '+' '-' '-' '-' '-' '+' '+' '-' '-' '-' '-' '-'
 '+' '+' '+' '+' '+' '-' '+' '-' '-' '+' '+' '-' '-' '+' '-' '-' '-' '-'
 '-' '-' '+' '+' '+' '+' '-' '+' '-' '-' '+' '-' '+' '+' '-' '+' '-' '+'
 '-' '-' '+' '+' '-' '-' '+' '+' '+' '-' '-' '-' '-' '-' '-' '+' '-' '+'
 '-' '+' '-' '-' '-' '-' '-' '+' '+' '+' '+' '-' '+' '-' '-' '+' '+' '+'
 '-' '-' '+' '-' '-' '+' '+' '-' '-' '+' '+' '-' '+' '+' '-' '-' '+' '-'
 '-' '+' '+' '-' '-' '+' '-' '-' '-' '+' '+' '-' '+' '-' '+' '-' '-' '-'
 '-' '+' '-' '-' '-' '-' '-' '-' '+' '-' '-' '+']


**MODEL EVALUATION**

In [22]:
# Logistic Regression Model Evaluation
log_acc = accuracy_score(y_test, y_pred_log) 
log_precision = precision_score(y_test, y_pred_log) 
log_recall = recall_score(y_test, y_pred_log) 
log_f1 = f1_score(y_test, y_pred_log) 
log_cm = confusion_matrix(y_test, y_pred_log) 

ValueError: pos_label=1 is not a valid label. It should be one of ['+', '-']

In [ ]:
# Decision Tree Model Evaluation
decision_acc = accuracy_score(y_test, y_pred_decision) 
decision_precision = precision_score(y_test, y_pred_decision) 
decision_recall = recall_score(y_test, y_pred_decision) 
decision_f1 = f1_score(y_test, y_pred_decision) 
decision_cm = confusion_matrix(y_test, y_pred_decision) 

In [ ]:
# Random Forest Model Evaluation
random_acc = accuracy_score(y_test, y_pred_random) 
random_precision = precision_score(y_test, y_pred_random) 
random_recall = recall_score(y_test, y_pred_random) 
random_f1 = f1_score(y_test, y_pred_random) 
random_cm = confusion_matrix(y_test, y_pred_random) 

In [ ]:
# Printing the result for the model evaluation
print("="*20)
print("MODEL EVALUATION")
print("="*20)
print("LOGISTIC REGRESSION")
print(f"Accuracy: {log_acc:.4f}")
print(f"Precision: {log_precision:.4f}") 
print(f"Recall: {log_recall:.4f}") 
print(f"F1-Score: {log_f1:.4f}")
print(f"Confusion Matrix:") 
print(log_cm)
print("-"*20)

print("DECISION TREE CLASSIFIER")
print(f"Accuracy: {decision_acc:.4f}")
print(f"Precision: {decision_precision:.4f}") 
print(f"Recall: {decision_recall:.4f}") 
print(f"F1-Score: {decision_f1:.4f}")
print(f"Confusion Matrix:") 
print(decision_cm)
print("-"*20)

print("RANDOM FOREST CLASSIFIER")
print(f"Accuracy: {random_acc:.4f}")
print(f"Precision: {random_precision:.4f}") 
print(f"Recall: {random_recall:.4f}") 
print(f"F1-Score: {random_f1:.4f}")
print(f"Confusion Matrix:") 
print(random_cm)

**BEST MODEL WITH ACCURACY**

In [ ]:
# Creating dictionary of accuracies
accuracies = {
    'Logistic Regression': log_acc,
    'Decision Tree': decision_acc,
    'Random Forest': random_acc
}

In [ ]:
# Checking for the best model and accuraxy

best_model_name = max(accuracies, key=accuracies.get) 
best_accuracy = accuracies[best_model_name] 

if best_model_name == 'Logistic Regression':
    best_model = logistic_grid.best_estimator_
elif best_model_name == 'Decision Tree':
    best_model = decision_grid.best_estimator_ 
else:
    best_model = random_grid.best_estimator_ 

print(f"BEST MODEL: {best_model_name}")
print(f"BEST ACCURACY: {best_accuracy:.4f}") 
print(f"BEST MODRL = {best_model}")
print(f"BEST SCORE = {best_accuracy:.4f}") 


if best_accuracy >= 0.75:
    print(f"Requirement met (Accuracy >= 0.75).") 
else:
    print(f"Does not met the required value.")